# Target classification: zero-shot / few-shot

Chạy cùng `src/inference.py`, output JSON chỉ có `target` và `reasoning`. Cần dữ liệu TSV riêng tại đường dẫn trong config, môi trường Python đã cài `requirements.txt`, và GPU phù hợp. Notebook không chứa dữ liệu hoặc output mẫu thật.


In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, cwd / 'task 8', cwd.parent)
             if (p / 'src/inference.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from task 8, task 8/notebooks, or the repository root.')
sys.path.insert(0, str(ROOT / 'src'))
from inference import (INPUT_VARIANTS, load_prompt, make_messages, print_metrics,
                       read_tsv, validate_demonstrations)
print(ROOT)


Chọn model, cấu hình đầu vào, chế độ prompt và split. Các thay đổi dưới đây chỉ ghi vào config riêng trong `outputs/notebook/`. `RUN_INFERENCE = True` cho phép cell chạy tải model và suy luận.


In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
INPUT_VARIANT = 'comment_title_desc'
PROMPT_MODE = 'zero_shot'  # zero_shot or few_shot
SPLIT = 'dev'  # train, dev, test; few-shot on train is rejected for overlap
SHOTS = 6
RUN_INFERENCE = False

config = json.loads((ROOT / 'src/configs/inference.json').read_text(encoding='utf-8'))
config.update(model_id=MODEL_ID, input_variant=INPUT_VARIANT)
config['prompt'].update(mode=PROMPT_MODE, shots=SHOTS)
config['data']['split'] = SPLIT
config['generation']['max_new_tokens'] = 256
# Adjust model dtype/device_map/max_memory/GPTQ backend here if needed.
assert INPUT_VARIANT in INPUT_VARIANTS
assert PROMPT_MODE in ('zero_shot', 'few_shot')
assert SPLIT in ('train', 'dev', 'test')
display(config)


Kiểm tra dữ liệu và few-shot trước khi load model. Chỉ hiện tên trường và số messages; không in nội dung dữ liệu vào notebook.


In [ ]:
rows = read_tsv(ROOT / config['data']['files'][SPLIT])
filename = 'fs_prompt.md' if PROMPT_MODE == 'few_shot' else 'zs_prompt.md'
blocks, examples = load_prompt(ROOT / config['prompt']['directory'] / INPUT_VARIANT / filename, PROMPT_MODE)
if PROMPT_MODE == 'few_shot':
    if not 1 <= SHOTS <= len(examples):
        raise ValueError('Invalid SHOTS for this prompt.')
    examples = examples[:SHOTS]
    validate_demonstrations(examples, read_tsv(ROOT / config['data']['files']['train']), rows, INPUT_VARIANT)
messages = make_messages(blocks, examples, rows[0], config['data']['language'], INPUT_VARIANT)
print({'rows': len(rows), 'input_fields': INPUT_VARIANTS[INPUT_VARIANT],
       'demonstrations': len(examples), 'messages_per_query': len(messages),
       'output_schema': ['target', 'reasoning']})


In [ ]:
if not RUN_INFERENCE:
    raise RuntimeError('Set RUN_INFERENCE = True after reviewing the configuration.')
work_dir = ROOT / 'outputs/notebook'
work_dir.mkdir(parents=True, exist_ok=True)
config_path = work_dir / 'inference.json'
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
output_root = ROOT / config['output_dir']
before = set(output_root.rglob('run.json'))
subprocess.run([sys.executable, str(ROOT / 'src/inference.py'), '--config', str(config_path)],
               cwd=ROOT, check=True)
created = set(output_root.rglob('run.json')) - before
matches = [p for p in created if json.loads(p.read_text(encoding='utf-8'))['config'] == config]
if len(matches) != 1:
    raise RuntimeError(f'Expected one new matching run; found {len(matches)}. Inspect outputs manually.')
run_dir = matches[0].parent
print(run_dir)


Xem metrics và thống kê lỗi định dạng. Nhãn vẫn được chấm độc lập với lỗi schema; `raw_output` được giữ trong TSV để kiểm tra. Không lưu output notebook chứa dữ liệu trước khi commit.


In [ ]:
report = json.loads((run_dir / 'metrics.json').read_text(encoding='utf-8'))
print_metrics(report)
with (run_dir / 'predictions.tsv').open(encoding='utf-8', newline='') as stream:
    predictions = list(csv.DictReader(stream, delimiter='\t'))
print({'predictions': len(predictions),
       'schema_errors': sum(p['annotation_valid'] != 'True' for p in predictions),
       'missing_reasons': sum(not p['reasoning'].strip() for p in predictions)})
print('Predictions:', run_dir / 'predictions.tsv')
